Check model predictions - RoBERTa (5 epochs, early stopping, seed)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import random 

from transformers import (
    RobertaTokenizer, 
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from datasets import Dataset as HFDataset, DatasetDict

In [ ]:
def load_dataset(clean=True):

    
    with open("../data/train_data.json", "r") as f:
        train = json.load(f)
    with open("../data/val_data.json", "r") as f:
        val = json.load(f)
    with open("../data/test_data.json", "r") as f:
        test = json.load(f)
    
    train_df = pd.DataFrame(train["data"])
    val_df = pd.DataFrame(val["data"])
    test_df = pd.DataFrame(test["data"])
    
    if clean:
        train_len_before = len(train_df)
        val_len_before = len(val_df)
        test_len_before = len(test_df)
        
        train_df = train_df.dropna(subset=['text'])
        val_df = val_df.dropna(subset=['text'])
        test_df = test_df.dropna(subset=['text'])
        
        train_removed = train_len_before - len(train_df)
        val_removed = val_len_before - len(val_df)
        test_removed = test_len_before - len(test_df)
        
        print(f"Removed {train_removed} rows with NaN values from training dataset")
        print(f"Removed {val_removed} rows with NaN values from validation dataset")
        print(f"Removed {test_removed} rows with NaN values from testing dataset")
        print(f"Total removed: {train_removed + val_removed + test_removed} rows")
    
    return {'train': train_df, 'val': val_df, 'test': test_df}

In [ ]:
dataset = load_dataset(clean = True)

In [ ]:
LABEL_MAP = {
    'neutral': 0,
    'moderate': 1,
    'high-risk': 2
}

EMOTION_INDEX = {
    0: 'neutral',
    1: 'moderate',
    2: 'high-risk'
}

In [ ]:
class BERTDataset(Dataset):
    
    def __init__(self, df, tokenizer, max_length, datacolumn, labelcolumn):
        self.data = df
        self.texts = self.data[datacolumn].tolist()
        self.labels = self.data[labelcolumn].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        if isinstance(label, str):
            label = LABEL_MAP[label]
        
        encoding = self.tokenizer.encode_plus(
            text,
            max_length=self.max_length,
            add_special_tokens=True,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long),
        }


def create_batch_tokenized_datasets_roberta(max_length=512, model_name="roberta-base"):
    dataframes = load_dataset(clean=True)
    hf_datasets = {}
    
    for split, df in dataframes.items():
        df = df.copy()
        df['label'] = df['my_label'].map(LABEL_MAP)
        hf_datasets[split] = HFDataset.from_pandas(df)
    
    dataset_dict = DatasetDict(hf_datasets)
    roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    
    def tokenize(batch):
        return roberta_tokenizer(
            batch["text"],
            padding="max_length",
            max_length=max_length,
            truncation=True,
            return_tensors="pt"
        )
    
    dataset_encoded = dataset_dict.map(tokenize, batched=True, batch_size=None)
    dataset_encoded.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "label"]
    )
    
    print(f"Created batch-tokenized datasets:")
    for split, dataset in dataset_encoded.items():
        print(f"  {split}: {len(dataset)} examples")
    
    return dataset_encoded


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
   
    accuracy = np.mean(predictions == labels)
    
    from sklearn.metrics import f1_score
    f1 = f1_score(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
    }

def setup_evaluation_args():
    return TrainingArguments(
        output_dir='../models/resultsroberta',
        eval_strategy="no", 
        per_device_eval_batch_size=16,
        no_cuda=True,
        use_mps_device=False,
        report_to=None,
        seed=42,
    )

In [ ]:
model_path = '../models/resultsroberta/checkpoint-516'
tokenizer_path = '../models/resultsroberta'
tokenizer = RobertaTokenizer.from_pretrained(tokenizer_path)
model = RobertaForSequenceClassification.from_pretrained(model_path)

In [ ]:
training_args = setup_evaluation_args()
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
)

Evaluate predictions on the test set

In [ ]:
dataset_encoded = create_batch_tokenized_datasets_roberta(max_length=512, model_name="roberta-base")

In [ ]:
preds_output = trainer.predict(dataset_encoded["test"])

In [ ]:
preds_output.metrics

In [ ]:
y_valid = np.asarray(dataset_encoded["test"]["label"]) 
y_preds = np.argmax(preds_output.predictions, axis=1)

print('\nClassification Report for RoBERTa:\n')
print(classification_report(
    y_valid,
    y_preds,
    target_names=['neutral', 'moderate', 'high-risk']
))

Explore misclassified labels

In [ ]:
dataset["test"]

In [ ]:
predicted_labels = np.argmax(preds_output.predictions, axis=1)

true_labels_str = dataset["test"]["my_label"]  
true_labels = [LABEL_MAP[label] for label in true_labels_str]


df = pd.DataFrame({
    "text": dataset["test"]["text"],             
    "true_label": true_labels,   
    "predicted_label_roberta": predicted_labels
})

In [ ]:
df

High risk classified as neutral

In [ ]:
df["high_risk_to_neutral"] = (
    (df["true_label"] == 2) & (df["predicted_label_roberta"] == 0)
)

Neutral classified as high risk 

In [ ]:
df["neutral_to_high_risk"] = (
    (df["true_label"] == 0) & (df["predicted_label_roberta"] == 2)
)

In [ ]:
print(df[df["high_risk_to_neutral"] == True])

In [ ]:
print(df[df["neutral_to_high_risk"] == True])